# 02 – Regime Detection

Fit Hidden Markov Models and Gaussian Mixture Models to identify distinct market regimes from credit-spread data.

**Covers:** HMM fitting, regime labelling, transition matrices, regime-conditional statistics.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from config.settings import DEFAULT_START_DATE, DEFAULT_END_DATE, FRED_API_KEY, DATA_DIR, HMM_N_STATES
from src.data.fetcher import fetch_all_data

df = fetch_all_data(
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
    api_key=FRED_API_KEY,
    cache_dir=DATA_DIR,
)
print(f'Data loaded: {df.shape}')

In [ ]:
from src.models.regime import fit_hmm, label_regimes, get_transition_matrix, compute_regime_stats

# Use HY spread + VIX as inputs
feature_cols = [c for c in ['hy_spread', 'vix'] if c in df.columns]
data = df[feature_cols].dropna()

# Fit HMM
model = fit_hmm(data.values, n_states=HMM_N_STATES)
regimes = label_regimes(model, data.values, model_type='hmm')
print('Regime counts:', pd.Series(regimes).value_counts().sort_index().to_dict())

In [ ]:
# Transition matrix
trans = get_transition_matrix(model, model_type='hmm')
print('Transition Matrix:')
trans.round(3)

In [ ]:
# Regime-conditional statistics
df_valid = df.loc[data.index]

equity_col = 'sp500_return' if 'sp500_return' in df_valid.columns else None
if equity_col:
    stats = compute_regime_stats(df_valid, regimes, equity_col=equity_col, spread_col='hy_spread')
    print('Regime Stats:')
    display(stats.round(4))

In [ ]:
# Regime overlay chart
from src.visualization.plots import plot_regime_overlay

fig = plot_regime_overlay(df_valid, regimes, spread_col='hy_spread')
fig.show()

In [ ]:
# Also try GMM for comparison
from src.models.regime import fit_gmm

gmm_model = fit_gmm(data.values, n_components=HMM_N_STATES)
gmm_regimes = label_regimes(gmm_model, data.values, model_type='gmm')
print('GMM Regime counts:', pd.Series(gmm_regimes).value_counts().sort_index().to_dict())